# Graph-based Drug Repurposing using PrimeKG.

### Team - 6.
-------------------------------------------------------------------
* #### Contributors :

1. Athul Dinesh
2. Dilep Shetty Ittanguru Venkatesh
3. Stuti Jandhyala

-------------------------------------------------------------------


* This notebook documents the end-to-end workflow for building a biomedical knowledge graph from PrimeKG and using it for drug repurposing experiments. The workflow covers data preparation, graph construction, model training, evaluation, evidence-aware reranking, and a final demo query.

## 1. Environment setup

This section installs the required graph learning libraries, imports the packages used throughout the notebook, and prints a quick environment check so the runtime can be verified before loading data.


In [ ]:
# BLOCK 01 - Environment Setup

# Install PyTorch

!pip install --quiet torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 \
    --index-url https://download.pytorch.org/whl/cu128

# Install PyG sparse/scatter extensions

!pip install --quiet pyg-lib \
    -f https://data.pyg.org/whl/torch-2.8.0+cu128.html

import torch

!pip install --quiet \
    torch-scatter \
    torch-sparse \
    torch-cluster \
    -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Install PyG
!pip install --quiet git+https://github.com/pyg-team/pytorch_geometric.git

# Standard libraries
import warnings
import sys
import platform
import random
import copy
import math
import re
import difflib
import json
from pathlib import Path
from copy import deepcopy
from collections import defaultdict, Counter
from datetime import datetime

warnings.filterwarnings("ignore")

# Dataframe and Display imports
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows",    20)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width",       200)

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import networkx as nx
import textwrap

# Metrics and ML utilities
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef,
    log_loss,
    ndcg_score,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve

# PyTorch + PyG
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch_geometric
import torch_scatter
import torch_sparse
import torch_cluster

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, HGTConv, Linear

try:
    import pyg_lib
    _pyg_lib_version = pyg_lib.__version__
except ImportError:
    _pyg_lib_version = "not installed"

# Google Colab utilities
from google.colab import drive
from IPython.display import Image, display

# Global reproducibility seed
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)


# Environment summary
print("=" * 55)
print("  ENVIRONMENT SUMMARY")
print("=" * 55)
print(f"  Python          : {sys.version.split()[0]}")
print(f"  Platform        : {platform.platform()}")
print()
print(f"  PyTorch         : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA version    : {torch.version.cuda}")
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
print()
print(f"  PyG             : {torch_geometric.__version__}")
print(f"  torch-scatter   : {torch_scatter.__version__}")
print(f"  torch-sparse    : {torch_sparse.__version__}")
print(f"  torch-cluster   : {torch_cluster.__version__}")
print(f"  pyg-lib         : {_pyg_lib_version}")
print()

# Select device : used globally throughout the notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Active device   : {device}")
print("=" * 55)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 116.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 135.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  ENVIRONMENT SUMMARY
  Python          : 3.12.13
  Platform        : Linux-6.6.113+-x86_64-with-glibc2.35

  PyTorch         : 2.8.0+cu128
  CUDA 